# Notebook 10 — Portfolio Construction

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 10 of 12  
**Time estimate:** 60 minutes

> A portfolio is not a collection of files. It is a curated argument that the owner
> can do real computational biology work. This notebook audits the repository,
> produces a completeness scorecard, and generates a polished portfolio page draft.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Dict

# ---- Portfolio completeness criteria ----

@dataclass
class PortfolioCriteria:
    """One criterion in the portfolio audit."""
    name: str
    category: str          # 'code', 'science', 'communication', 'openness'
    weight: float          # relative importance 1-3
    description: str
    done: bool = False
    notes: str = ''

PORTFOLIO_CRITERIA: List[PortfolioCriteria] = [
    # Code quality
    PortfolioCriteria('Clean README',          'code', 3, 'Top-level README explains the project in < 300 words, has badges (CI, license)', done=True),
    PortfolioCriteria('Working install',       'code', 3, 'One command (`pip install -e .` or `conda env create`) reproduces the environment', done=True),
    PortfolioCriteria('CI passes',             'code', 2, 'GitHub Actions green badge on main branch', done=True),
    PortfolioCriteria('Tests exist',           'code', 2, 'At least 10 pytest tests for compbio_utils; all pass', done=True),
    PortfolioCriteria('Docstrings',            'code', 1, 'All public functions/classes have NumPy-style docstrings', done=True),
    PortfolioCriteria('No secrets in repo',    'code', 3, 'No API keys, tokens, passwords, or credentials committed', done=True),
    # Scientific content
    PortfolioCriteria('CP01 RNA-seq',          'science', 3, 'Negative binomial DE analysis, BH correction, heatmap; results paragraph written', done=True),
    PortfolioCriteria('CP02 TF binding',       'science', 3, 'CNN vs k-mer baseline, AUROC comparison, Wilcoxon test, abstract written', done=True),
    PortfolioCriteria('CP03 Epidemic models',  'science', 3, 'ODE vs Network vs Spatial, MCMC, AIC, 4-panel figures', done=True),
    PortfolioCriteria('Paper notes',           'science', 2, '≥ 5 papers in paper-notes/ with Pass-3 reconstruction notes', done=False),
    PortfolioCriteria('Module diversity',      'science', 2, '≥ 5 modules represented in portfolio/ folder', done=True),
    # Communication
    PortfolioCriteria('Portfolio page',        'communication', 3, 'portfolio/ folder with polished, linked README or index.md', done=False),
    PortfolioCriteria('Figures publication-ready', 'communication', 2, 'All capstone figures use PUB_RCPARAMS (dpi=300, no top/right spines)', done=True),
    PortfolioCriteria('Track A narrative',     'communication', 2, 'Can explain every module in one sentence for an interview', done=False),
    PortfolioCriteria('Commit history clean',  'communication', 1, 'Commit messages are descriptive; no "WIP", "fix", or "misc" without context', done=True),
    # Open science
    PortfolioCriteria('License',               'openness', 3, 'MIT license in repo root', done=True),
    PortfolioCriteria('DOI / Zenodo',          'openness', 2, 'Zenodo archive created (generates a citable DOI)', done=False),
    PortfolioCriteria('ORCID linked',          'openness', 2, 'ORCID in GitHub profile and any preprint author list', done=False),
    PortfolioCriteria('Data provenance',       'openness', 2, 'Every dataset has a datasets/README.md with source URL and access date', done=True),
    PortfolioCriteria('Availability statement','openness', 2, 'Code availability section in every capstone writeup', done=True),
]

# ---- Portfolio audit ----
def portfolio_score(criteria: List[PortfolioCriteria]) -> Dict:
    by_cat = {}
    for c in criteria:
        by_cat.setdefault(c.category, []).append(c)
    result = {}
    for cat, items in by_cat.items():
        done_w = sum(c.weight for c in items if c.done)
        total_w = sum(c.weight for c in items)
        result[cat] = {'score': done_w / total_w, 'done': sum(1 for c in items if c.done), 'total': len(items)}
    overall_done = sum(c.weight for c in criteria if c.done)
    overall_total = sum(c.weight for c in criteria)
    result['overall'] = {'score': overall_done / overall_total, 'done': sum(1 for c in criteria if c.done), 'total': len(criteria)}
    return result

scores = portfolio_score(PORTFOLIO_CRITERIA)
print('=== Portfolio Completeness Audit ===')
for cat, s in scores.items():
    bar = '█' * int(s['score'] * 20) + '░' * (20 - int(s['score'] * 20))
    print(f'{cat:<18} [{bar}] {s["score"]:>5.0%}  ({s["done"]}/{s["total"]} criteria)')

print('\nOpen items:')
for c in PORTFOLIO_CRITERIA:
    if not c.done:
        print(f'  [ ] [{c.category}] {c.name}: {c.description}')

In [ ]:
# ---- GitHub portfolio page draft ----

PORTFOLIO_PAGE = '''# Vinoth Kumar — Computational Biology Portfolio

**Track A target:** Research Associate / JRF / Bioinformatics Engineer (India, 2027)  
**Track B target:** Fully-funded European PhD in Computational Biology (2027 intake)  
**Repository:** github.com/Vinoth-ai-20/computational-biology  
**Contact:** vinoth.ac.in@gmail.com | ORCID: [link]

---

## Highlighted Projects

### CP01 — RNA-seq Differential Expression Analysis
- Negative-binomial count data (5,000 genes × 6 samples); 300 true positives injected
- Implemented median-of-ratios normalization (DESeq2-style) from scratch in NumPy
- Benjamini–Hochberg FDR correction implemented from scratch; 15% FDR threshold
- Result: 287/300 true DE genes recovered (sensitivity 95.7%)
- `curriculum/20_capstone_projects/notebooks/02_cp01_data_generation_qc.ipynb`

### CP02 — Transcription Factor Binding Site Prediction
- GATAAG motif benchmark: N=3,000 sequences (L=101 bp); 5-fold stratified CV
- Baseline: k-mer (k=4) features + SVM (AUROC 0.910 ± 0.003)
- Model: TFBindingCNN (Conv1d → BN → AdaptiveMaxPool → FC) in PyTorch
- Result: AUROC 0.983 ± 0.003 (+7.3 pp vs SVM, Wilcoxon p<0.001)
- `curriculum/20_capstone_projects/notebooks/06_cp02_cnn_implementation.ipynb`

### CP03 — Epidemic Model Comparison
- Three SIR variants: ODE (deterministic), Network (Erdős–Rényi), Spatial ABM (2D grid)
- MCMC parameter estimation (Metropolis–Hastings, 3,000 samples, Poisson likelihood)
- Profile likelihood 95% CI; AIC model comparison (SIR vs exponential growth)
- `curriculum/20_capstone_projects/notebooks/08_cp03_model_implementation.ipynb`

---

## Skills Demonstrated

| Domain | Tools & Methods |
|---|---|
| RNA-seq | Negative binomial models, DESeq2-style normalization, BH correction, volcano/MA plots |
| Sequence analysis | k-mer features, CNNs for sequences, motif discovery |
| Epidemiology | ODE integration, stochastic ABM, MCMC, AIC model selection |
| ML/DL | scikit-learn, PyTorch, 5-fold CV, AUROC/AUPRC, early stopping |
| Statistics | Wilcoxon signed-rank, bootstrap CI, FDR correction, power analysis |
| Software | Python packaging, pytest, Sphinx, Click CLI, GitHub Actions CI/CD |
| HPC | NumPy vectorization, multiprocessing, Numba JIT, SLURM, Rust (PyO3) |
| Writing | IMRaD structure, 6-sentence abstract, publication figures (300 dpi) |

---

## Module Completion Summary

See `COMPUTATIONAL_BIOLOGY_LEARNING_PROGRESS.md` for the live tracker.
'''

# Count words in portfolio page draft
word_count = len(PORTFOLIO_PAGE.split())
print(f'Portfolio page draft: {word_count} words')
print('First 300 chars:', PORTFOLIO_PAGE[:300])

In [ ]:
# ---- Visualization: portfolio radar + completeness bar ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Category scores as bar chart
ax = axes[0]
cats = ['code', 'science', 'communication', 'openness']
cat_scores = [scores[c]['score'] for c in cats]
cat_colors = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759']
bars = ax.barh(cats, cat_scores, color=cat_colors, edgecolor='black', alpha=0.8)
for bar, score, cat in zip(bars, cat_scores, cats):
    ax.text(score + 0.01, bar.get_y() + bar.get_height()/2,
             f'{score:.0%} ({scores[cat]["done"]}/{scores[cat]["total"]})',
             va='center', fontsize=9)
ax.axvline(0.8, color='grey', ls='--', lw=1, label='80% threshold')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Weighted score')
ax.set_title(f'A. Portfolio completeness by category\n(Overall: {scores["overall"]["score"]:.0%})')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel B: Criteria grid (done vs not done)
ax = axes[1]
cat_order = ['code', 'science', 'communication', 'openness']
row_labels = [f'[{c.category[:3].upper()}] {c.name}' for c in PORTFOLIO_CRITERIA]
done_flags = [1 if c.done else 0 for c in PORTFOLIO_CRITERIA]
weights    = [c.weight for c in PORTFOLIO_CRITERIA]

y_pos = np.arange(len(PORTFOLIO_CRITERIA))
colors_row = ['#59a14f' if d else '#e15759' for d in done_flags]
bars2 = ax.barh(y_pos, weights, color=colors_row, edgecolor='white', height=0.7, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(row_labels, fontsize=7)
ax.set_xlabel('Criterion weight (1=low, 3=high)')
ax.set_title('B. All criteria: done (green) vs pending (red)')
done_patch   = mpatches.Patch(color='#59a14f', alpha=0.85, label='Done')
pending_patch = mpatches.Patch(color='#e15759', alpha=0.85, label='Pending')
ax.legend(handles=[done_patch, pending_patch], fontsize=8, loc='lower right')
ax.text(-0.12, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold')

plt.suptitle('Module 20: Portfolio Construction Audit', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('portfolio_audit.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Write two sentences: (1) what does the portfolio currently demonstrate
> most strongly? (2) What is the single most important pending item to
> complete before submitting a Track A application?]*

---
*Next: `11_phd_application_package.ipynb`*